# NovaCart — Silver Products Transformation

## 1. Import Libraries

In [0]:
from pyspark.sql import functions as F

## 2. Define Storage Paths

In [0]:
BRONZE_PRODUCTS_PATH = (
    "abfss://bronze@stnovacartdev.dfs.core.windows.net/"
    "olist/products"
)

SILVER_PRODUCTS_PATH = (
    "abfss://silver@stnovacartdev.dfs.core.windows.net/"
    "olist/products"
)

QUARANTINE_PRODUCTS_PATH = (
    "abfss://quarantine@stnovacartdev.dfs.core.windows.net/"
    "olist/products"
)

print(f"Bronze path: {BRONZE_PRODUCTS_PATH}")
print(f"Silver path: {SILVER_PRODUCTS_PATH}")
print(f"Quarantine path: {QUARANTINE_PRODUCTS_PATH}")

## 3. Read Bronze Products Data

In [0]:
products_bronze_df = (
    spark.read
    .format("delta")
    .load(BRONZE_PRODUCTS_PATH)
)

bronze_row_count = products_bronze_df.count()

print("Bronze products loaded successfully.")
print(f"Bronze row count: {bronze_row_count}")

products_bronze_df.printSchema()
display(products_bronze_df.limit(10))

## 4. Validate Required Columns

In [0]:
required_columns = [
    "product_id",
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
    "_source_file",
    "_ingestion_timestamp",
    "_batch_id",
]

missing_columns = [
    column
    for column in required_columns
    if column not in products_bronze_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required Bronze columns: {missing_columns}"
    )

print("Required-column validation passed.")

## 5. Profile Missing and Invalid Product Values

In [0]:
products_profile_df = products_bronze_df.agg(
    F.count("*").alias("total_rows"),

    F.sum(
        (
            F.col("product_id").isNull()
            | (F.trim(F.col("product_id")) == "")
        ).cast("int")
    ).alias("invalid_product_id"),

    F.sum(
        (
            F.col("product_category_name").isNull()
            | (F.trim(F.col("product_category_name")) == "")
        ).cast("int")
    ).alias("missing_product_category_name"),

    F.sum(F.col("product_name_lenght").isNull().cast("int"))
        .alias("missing_product_name_length"),

    F.sum((F.col("product_name_lenght") < 0).cast("int"))
        .alias("negative_product_name_length"),

    F.sum(F.col("product_description_lenght").isNull().cast("int"))
        .alias("missing_product_description_length"),

    F.sum((F.col("product_description_lenght") < 0).cast("int"))
        .alias("negative_product_description_length"),

    F.sum(F.col("product_photos_qty").isNull().cast("int"))
        .alias("missing_product_photos_qty"),

    F.sum((F.col("product_photos_qty") < 0).cast("int"))
        .alias("negative_product_photos_qty"),

    F.sum(F.col("product_weight_g").isNull().cast("int"))
        .alias("missing_product_weight_g"),

    F.sum((F.col("product_weight_g") <= 0).cast("int"))
        .alias("non_positive_product_weight_g"),

    F.sum(F.col("product_length_cm").isNull().cast("int"))
        .alias("missing_product_length_cm"),

    F.sum((F.col("product_length_cm") <= 0).cast("int"))
        .alias("non_positive_product_length_cm"),

    F.sum(F.col("product_height_cm").isNull().cast("int"))
        .alias("missing_product_height_cm"),

    F.sum((F.col("product_height_cm") <= 0).cast("int"))
        .alias("non_positive_product_height_cm"),

    F.sum(F.col("product_width_cm").isNull().cast("int"))
        .alias("missing_product_width_cm"),

    F.sum((F.col("product_width_cm") <= 0).cast("int"))
        .alias("non_positive_product_width_cm"),
)

display(products_profile_df)

## 6. Print Full Product Quality Profile

In [0]:
profile = products_profile_df.first().asDict()

for metric, value in profile.items():
    print(f"{metric}: {value}")

## 7. Check Duplicate Product IDs

In [0]:
duplicate_product_ids_df = (
    products_bronze_df
    .groupBy("product_id")
    .count()
    .filter(
        F.col("product_id").isNotNull()
        & (F.col("count") > 1)
    )
)

duplicate_product_id_count = duplicate_product_ids_df.count()

print(
    f"Number of product_id values appearing more than once: "
    f"{duplicate_product_id_count}"
)

display(duplicate_product_ids_df.limit(20))

## 9. Clean, Rename, and Standardize Product Fields

In [0]:
products_cleaned_df = (
    products_bronze_df
    .withColumn(
        "product_id",
        F.trim(F.col("product_id"))
    )
    .withColumn(
        "product_category_name",
        F.lower(F.trim(F.col("product_category_name")))
    )
    .withColumnRenamed(
        "product_name_lenght",
        "product_name_length"
    )
    .withColumnRenamed(
        "product_description_lenght",
        "product_description_length"
    )
)

## 10. Handle Missing Product Categories

In [0]:
products_cleaned_df = (
    products_cleaned_df
    .withColumn(
        "product_category_name",
        F.when(
            F.col("product_category_name").isNull()
            | (F.col("product_category_name") == ""),
            F.lit("unknown")
        ).otherwise(F.col("product_category_name"))
    )
)

## 11. Define Product Validation Rules

In [0]:
invalid_product_id_condition = (
    F.col("product_id").isNull()
    | (F.col("product_id") == "")
)

invalid_product_name_length_condition = (
    F.col("product_name_length").isNotNull()
    & (F.col("product_name_length") < 0)
)

invalid_product_description_length_condition = (
    F.col("product_description_length").isNotNull()
    & (F.col("product_description_length") < 0)
)

invalid_product_photos_qty_condition = (
    F.col("product_photos_qty").isNotNull()
    & (F.col("product_photos_qty") < 0)
)

invalid_product_weight_condition = (
    F.col("product_weight_g").isNull()
    | (F.col("product_weight_g") <= 0)
)

invalid_product_length_condition = (
    F.col("product_length_cm").isNull()
    | (F.col("product_length_cm") <= 0)
)

invalid_product_height_condition = (
    F.col("product_height_cm").isNull()
    | (F.col("product_height_cm") <= 0)
)

invalid_product_width_condition = (
    F.col("product_width_cm").isNull()
    | (F.col("product_width_cm") <= 0)
)

## 12. Assign Product Rejection Reasons

In [0]:
products_validated_df = products_cleaned_df.withColumn(
    "_rejection_reason",

    F.when(
        invalid_product_id_condition,
        F.lit("MISSING_PRODUCT_ID")
    )
    .when(
        invalid_product_name_length_condition,
        F.lit("INVALID_PRODUCT_NAME_LENGTH")
    )
    .when(
        invalid_product_description_length_condition,
        F.lit("INVALID_PRODUCT_DESCRIPTION_LENGTH")
    )
    .when(
        invalid_product_photos_qty_condition,
        F.lit("INVALID_PRODUCT_PHOTOS_QTY")
    )
    .when(
        invalid_product_weight_condition,
        F.lit("INVALID_PRODUCT_WEIGHT")
    )
    .when(
        invalid_product_length_condition,
        F.lit("INVALID_PRODUCT_LENGTH")
    )
    .when(
        invalid_product_height_condition,
        F.lit("INVALID_PRODUCT_HEIGHT")
    )
    .when(
        invalid_product_width_condition,
        F.lit("INVALID_PRODUCT_WIDTH")
    )
    .otherwise(F.lit(None))
)

## 13. Review Product Validation Results

In [0]:
display(
    products_validated_df
    .groupBy("_rejection_reason")
    .count()
    .orderBy("_rejection_reason")
)

## 14. Derive Product Volume

In [0]:
products_valid_df = (
    products_validated_df
    .filter(F.col("_rejection_reason").isNull())
    .drop("_rejection_reason")
)

products_silver_df = (
    products_valid_df
    .withColumn(
        "product_volume_cm3",
        (
            F.col("product_length_cm")
            * F.col("product_height_cm")
            * F.col("product_width_cm")
        )
    )
    .withColumn(
        "_silver_processed_at",
        F.current_timestamp()
    )
)

## 15. Prepare Quarantine Records

In [0]:
products_quarantine_df = (
    products_validated_df
    .filter(F.col("_rejection_reason").isNotNull())
    .withColumn(
        "_quarantined_at",
        F.current_timestamp()
    )
    .withColumn(
        "_source_dataset",
        F.lit("products")
    )
)

## 16. Count Silver and Quarantine Records

In [0]:
valid_row_count = products_silver_df.count()
quarantine_row_count = products_quarantine_df.count()

print(f"Valid Silver rows: {valid_row_count}")
print(f"Quarantined rows: {quarantine_row_count}")
print(f"Bronze input rows: {bronze_row_count}")

## 17. Validate Row-Count Reconciliation

In [0]:
if valid_row_count + quarantine_row_count != bronze_row_count:
    raise ValueError(
        "Row-count validation failed: "
        "Silver rows + quarantine rows do not equal Bronze input rows."
    )

print("Row-count validation passed.")

## 18. Write Valid Products to Silver

In [0]:
(
    products_silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_PRODUCTS_PATH)
)

print("Silver products written successfully.")

## 19. Write Invalid Products to Quarantine

In [0]:
(
    products_quarantine_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(QUARANTINE_PRODUCTS_PATH)
)

print("Products quarantine output written successfully.")

## 20. Read Written Delta Outputs

In [0]:
products_silver_written_df = (
    spark.read
    .format("delta")
    .load(SILVER_PRODUCTS_PATH)
)

products_quarantine_written_df = (
    spark.read
    .format("delta")
    .load(QUARANTINE_PRODUCTS_PATH)
)

silver_written_count = products_silver_written_df.count()
quarantine_written_count = products_quarantine_written_df.count()

print(f"Written Silver rows: {silver_written_count}")
print(f"Written quarantine rows: {quarantine_written_count}")

## 21. Validate Written Outputs

In [0]:
if silver_written_count != valid_row_count:
    raise ValueError(
        "Silver write validation failed: "
        f"expected {valid_row_count}, wrote {silver_written_count}."
    )

if quarantine_written_count != quarantine_row_count:
    raise ValueError(
        "Quarantine write validation failed: "
        f"expected {quarantine_row_count}, wrote "
        f"{quarantine_written_count}."
    )

if silver_written_count + quarantine_written_count != bronze_row_count:
    raise ValueError(
        "Final reconciliation failed: "
        "Silver + quarantine does not equal Bronze."
    )

print("Silver products pipeline completed successfully.")
print("Final row-count validation passed.")

## 22. Inspect Final Silver Products Dataset

In [0]:
products_silver_written_df.printSchema()

display(
    products_silver_written_df.select(
        "product_id",
        "product_category_name",
        "product_name_length",
        "product_description_length",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
        "product_volume_cm3",
        "_silver_processed_at"
    ).limit(20)
)